# P3-01 — Fonction de chargement commune des données

## Pourquoi une fonction commune ?

Jusqu'ici, chacun chargeait les fichiers à sa façon (`pd.read_csv("fichier.csv")` tout simple). Le problème : pandas devine automatiquement les types de chaque colonne, et il devine presque toujours trop large (`int64` au lieu de `int8`/`int16`, `object` au lieu de `category`). Sur un petit fichier ça ne change rien, mais sur `transactions_data.csv` (13 millions de lignes), ça peut faire la différence entre un notebook fluide et un notebook qui plante par manque de mémoire.

L'objectif de cette tâche est donc d'avoir **un seul code de chargement, partagé par toute l'équipe**, pour que :
- tout le monde travaille avec les mêmes types de données dès le départ
- on évite que chacun réinvente sa propre façon de lire les fichiers
- les futures étapes de nettoyage (P3-02, P3-03...) partent toutes de la même base

## Ce que fait la fonction

Le fichier `src/load_data.py` contient 4 fonctions : `load_transactions()`, `load_cards()`, `load_users()` et `load_mcc_codes()`.

Chacune fait deux choses :

1. **Lit le fichier par morceaux (chunks)** plutôt que d'un coup, pour limiter la charge mémoire pendant la lecture elle-même (surtout utile pour le fichier des transactions).
2. **Impose des types de données légers** dès la lecture, par exemple :
   - `int32` ou `int8` pour les identifiants et petits nombres (au lieu du `int64` par défaut)
   - `category` pour les colonnes répétitives comme `card_brand`, `merchant_city`, `errors` (au lieu de `object`), ce qui économise énormément de mémoire quand peu de valeurs différentes se répètent des millions de fois
   - `datetime` directement pour la colonne `date` des transactions

## Ce que la fonction ne fait PAS (volontairement)

Certaines colonnes restent **telles quelles pour l'instant** :
- Les montants en texte avec `$` (`amount`, `credit_limit`, `yearly_income`, etc.) → traités dans **P3-02**
- Les dates au format `mm/aaaa` (`expires`, `acct_open_date`) → traitées dans **P3-03**

**Pourquoi ne pas tout faire d'un coup ?** Pour que chaque transformation reste **séparée et traçable** dans le journal des transformations (P3-12). Si on mélange tout dans une seule fonction, on perd la trace de "qui a changé quoi et pourquoi" étape par étape.

## Comment l'utiliser

```python
from src.load_data import load_transactions, load_cards, load_users, load_mcc_codes

df_transaction = load_transactions()
df_card = load_cards()
df_users = load_users()
df_mcc = load_mcc_codes()
```

**Important** : à partir de maintenant, on importe ces fonctions plutôt que de refaire un `pd.read_csv()` à la main dans son propre notebook — sinon on perd l'intérêt d'avoir une base commune.

## Comment vérifier que ça fonctionne

Un petit contrôle rapide après chargement :

```python
for name, df in [("transactions", df_transaction), ("cards", df_card),
                  ("users", df_users), ("mcc", df_mcc)]:
    print(f"{name}: {df.shape[0]:,} lignes, {df.shape[1]} colonnes, "
          f"{df.memory_usage(deep=True).sum()/1e6:.1f} MB")
```

Si vous comparez avec une lecture "brute" (sans dtype précisé), vous devriez voir une réduction nette de la mémoire utilisée — c'est le signe que la fonction fait son travail.

In [ ]:
from src.load_data import load_transactions, load_cards, load_users, load_mcc_codes

In [ ]:
df_transaction = load_transactions()
df_card = load_cards()
df_users = load_users()
df_mcc = load_mcc_codes()

for name, df in [("transactions", df_transaction), ("cards", df_card),
                  ("users", df_users), ("mcc", df_mcc)]:
    print(f"{name}: {df.shape[0]:,} lignes, {df.shape[1]} colonnes, "
          f"{df.memory_usage(deep=True).sum()/1e6:.1f} MB")

# P3-02 — Conversion des montants en nombres

## Pourquoi cette étape ?

Plusieurs colonnes financières sont lues comme du **texte** (type `object`) parce qu'elles contiennent le symbole `$` (et parfois des virgules) :

| Fichier | Colonnes concernées |
|---|---|
| `transactions_data.csv` | `amount` |
| `cards_data.csv` | `credit_limit` |
| `users_data.csv` | `per_capita_income`, `yearly_income`, `total_debt` |

Tant qu'elles sont en texte, impossible de calculer une moyenne, une médiane, un écart au montant habituel (P4-04) ou de comparer deux montants. Il faut les convertir en nombres décimaux (`float`).

Attention au format des montants négatifs : le signe moins est écrit **après** le `$` (exemple : `$-77.00`). La conversion doit donc d'abord retirer le `$`, puis lire le nombre avec son signe.

## Où est le code ?

Dans un fichier séparé, `src/clean_data.py`, qui contient les fonctions de **transformation**. Le fichier `src/load_data.py` ne fait que **charger** les données. Cette séparation permet de tracer chaque étape dans le journal des transformations (P3-12).

## Fonction 1 : `convert_currency_column`

Elle prend une colonne texte et retourne une colonne numérique.

```python
def convert_currency_column(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")
```

### Explication ligne par ligne

| Syntaxe | Rôle |
|---|---|
| `series.astype(str)` | Force tout en texte pour que les opérations `.str` fonctionnent sur toutes les valeurs. Les valeurs manquantes deviennent le texte `"nan"`, qui sera reconverti en `NaN` à la dernière étape. |
| `.str.replace("$", "", regex=False)` | Supprime le symbole `$`. Le paramètre `regex=False` est important : en mode regex, `$` signifie « fin de chaîne » et ne retirerait rien. |
| `.str.replace(",", "", regex=False)` | Supprime les séparateurs de milliers (`1,234.56` devient `1234.56`). |
| `.str.strip()` | Retire les espaces au début et à la fin. |
| `pd.to_numeric(..., errors="coerce")` | Convertit en nombre. Toute valeur impossible à convertir devient `NaN` au lieu de provoquer une erreur. |

Les enchaînements `.str.replace(...).str.replace(...)` se lisent de haut en bas : chaque opération s'applique au résultat de la précédente.

### Exemples

| Texte d'origine | Résultat |
|---|---|
| `$14.57` | `14.57` |
| `$-77.00` | `-77.0` |
| `$24295` | `24295.0` |
| `$1,234.56` | `1234.56` |
| `abc` | `NaN` |

## Fonction 2 : `report_conversion_issues`

Le critère de fin de P3-02 est : **« Aucune valeur non convertie, ou cas listés »**. Cette fonction vérifie exactement cela.

```python
def report_conversion_issues(original: pd.Series, converted: pd.Series, label: str):
    failed_mask = converted.isna() & original.notna()
    n_failed = failed_mask.sum()
    print(f"{label}: {n_failed} valeur(s) non convertie(s)")
    if n_failed > 0:
        print(original[failed_mask].unique()[:10])
```

### Explication

| Syntaxe | Rôle |
|---|---|
| `converted.isna() & original.notna()` | Repère les lignes où la valeur **était présente** avant la conversion mais **est devenue vide** après. Ce sont les vraies erreurs de conversion. Une valeur déjà manquante au départ n'est pas comptée. |
| `failed_mask.sum()` | Compte les lignes en échec (`True` vaut 1, `False` vaut 0). |
| `original[failed_mask].unique()[:10]` | Affiche jusqu'à 10 valeurs distinctes qui ont posé problème, pour pouvoir les documenter. |

## Comment l'utiliser dans le notebook

On garde une copie de la colonne **avant** conversion, puis on compare :

```python
from src.clean_data import convert_currency_column, report_conversion_issues

original = df_transaction["amount"].copy()
df_transaction["amount"] = convert_currency_column(df_transaction["amount"])
report_conversion_issues(original, df_transaction["amount"], "transactions.amount")
```

Pour les trois colonnes de la table des clients, une boucle évite de répéter le code :

```python
for col in ["per_capita_income", "yearly_income", "total_debt"]:
    original = df_users[col].copy()
    df_users[col] = convert_currency_column(df_users[col])
    report_conversion_issues(original, df_users[col], f"users.{col}")
```

Le `.copy()` est nécessaire : sans lui, `original` pointerait vers la même colonne et serait modifié en même temps.

## Ce que cette étape ne fait PAS

- **Les montants négatifs ne sont pas corrigés ni supprimés.** Ils correspondent probablement à des remboursements. Leur traitement (garder, exclure ou marquer d'un indicateur) relève de **P3-08**.
- **Les revenus ou limites de crédit nuls ne sont pas modifiés.** Même règle : P3-08.
- **Les dates ne sont pas converties.** C'est le sujet de **P3-03**.

## À noter dans le journal des transformations (P3-12)

Pour chaque colonne : nombre de valeurs avant et après conversion, nombre de valeurs non converties (idéalement 0), et type de la colonne avant (`object`) et après (`float64`).

In [ ]:
from src.clean_data import convert_currency_column, report_conversion_issues

# transactions_data
original = df_transaction["amount"].copy()
df_transaction["amount"] = convert_currency_column(df_transaction["amount"])
report_conversion_issues(original, df_transaction["amount"], "transactions.amount")

# cards_data
original = df_card["credit_limit"].copy()
df_card["credit_limit"] = convert_currency_column(df_card["credit_limit"])
report_conversion_issues(original, df_card["credit_limit"], "cards.credit_limit")

# users_data
for col in ["per_capita_income", "yearly_income", "total_debt"]:
    original = df_users[col].copy()
    df_users[col] = convert_currency_column(df_users[col])
    report_conversion_issues(original, df_users[col], f"users.{col}")

In [ ]:
df_transaction.sample(5)   # 5 lignes au hasard